# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassantahir-afk/ML-Engineering-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** A Page is worth reviewing if it used to get a lot of traffic and its position got worse (slipped) between the first half and the second half
of the month.

**Reason Codes:** `traffic_position_decline` a page had meaningful traffic volume in the first half of the month, and its average search position worsened from the first half to the second half.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_frame = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    features AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS feat_impressions,
            SUM(CASE WHEN period = 'first_half' THEN gsc_clicks ELSE 0 END) AS feat_clicks,
            AVG(CASE WHEN period = 'first_half' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_first_half,
            AVG(CASE WHEN period = 'second_half' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_second_half,
            SUM(CASE WHEN period = 'first_half' THEN 1 ELSE 0 END) AS feat_days_active,
            SUM(CASE WHEN period = 'first_half' THEN gsc_clicks ELSE 0 END) * 1.0
                / NULLIF(SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END), 0) AS feat_ctr
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    ),
    label AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN period = 'second_half' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.feat_impressions,
        f.feat_clicks,
        f.position_first_half,
        f.position_second_half,
        (f.position_second_half - f.position_first_half) AS position_change,
        f.feat_days_active,
        f.feat_ctr,
        CASE
            WHEN (l.second_half - l.first_half) * 1.0 / NULLIF(l.first_half, 0) * 100 <= -10
            THEN TRUE ELSE FALSE
        END AS declining_flag
    FROM features f
    JOIN label l
        ON f.content_hash_id = l.content_hash_id AND f.client_hash_id = l.client_hash_id
    WHERE l.first_half > 0
      AND f.position_first_half IS NOT NULL
      AND f.position_second_half IS NOT NULL
""").df()

print("\n=== Feature Frame ===")
print(feature_frame.shape)
feature_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Feature Frame ===
(139747, 10)


,content_hash_id,client_hash_id,feat_impressions,feat_clicks,position_first_half,position_second_half,position_change,feat_days_active,feat_ctr,declining_flag
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,429.0,2.0,4.247255,4.532026,0.284771,15.0,0.004662,False
1,content_a7da352b73b02668,client_73cda7b4e4f265ea,2440.0,8.0,7.259861,7.230765,-0.029095,15.0,0.003279,False
2,content_d056587ff7faca0c,client_73cda7b4e4f265ea,1280.0,9.0,4.468441,4.450357,-0.018085,15.0,0.007031,False
3,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,19.0,0.0,11.050000,22.146296,11.096296,12.0,0.000000,False
4,content_2662845f598544ef,client_73cda7b4e4f265ea,97.0,0.0,8.765983,4.897222,-3.868761,15.0,0.000000,True


In [3]:
import pandas as pd

# Bucket by impression volume — using quantiles so buckets are roughly balanced in size
feature_frame['volume_bucket'] = pd.qcut(feature_frame['feat_impressions'], q=4, labels=['low', 'medium', 'high', 'very_high'], duplicates='drop')

volume_check = feature_frame.groupby('volume_bucket').agg(
    n=('declining_flag', 'count'),
    decline_rate=('declining_flag', 'mean')
)
print(volume_check)

                   n  decline_rate
volume_bucket                     
low            35381      0.282779
medium         34559      0.347145
high           34891      0.366255
very_high      34916      0.364618


/tmp/ipykernel_1247/683589305.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  volume_check = feature_frame.groupby('volume_bucket').agg(


**Volume:** **CONFIRMED**. Decline rate is consistently climbing across impression buckets (28.3% low, 34.7% medium, 36.6% high, 36.5% very_high) that means high-traffic pages are meaningfully more likely to be declining than low-traffic pages in this slice.

In [4]:
# Bucket by whether position got worse (positive change), stayed flat, or improved
feature_frame['position_slip_bucket'] = pd.cut(
    feature_frame['position_change'],
    bins=[-float('inf'), -2, 2, float('inf')],
    labels=['improved', 'flat', 'slipped']
)

position_check = feature_frame.groupby('position_slip_bucket').agg(
    n=('declining_flag', 'count'),
    decline_rate=('declining_flag', 'mean')
)
print(position_check)

                          n  decline_rate
position_slip_bucket                     
improved              37022      0.320728
flat                  58298      0.321006
slipped               44427      0.380940


/tmp/ipykernel_1247/1933563624.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  position_check = feature_frame.groupby('position_slip_bucket').agg(


**Position:  CONFIRMED** (moderate effect). Pages whose position slipped have a 38.1% decline rate, versus 32.1% for improved and 32.1% for flat. a real, directionally-correct signal, though a moderate one rather than an overwhelming one.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_frame['high_volume'] = (feature_frame['feat_impressions'] >= 500).astype(int)
feature_frame['position_slipped'] = (feature_frame['position_change'] > 2).astype(int)

feature_frame['score'] = (
    feature_frame['high_volume']
    * feature_frame['position_slipped']
    * feature_frame['feat_impressions']
)

feature_frame['reason_code'] = 'traffic_position_decline'
feature_frame['action'] = 'review'

# Only rows meeting both conditions actually get flagged with the reason/action;
# everything else stays score=0 with no action needed
feature_frame.loc[feature_frame['score'] == 0, ['reason_code', 'action']] = ['none', 'monitor']

ranked_queue = feature_frame.sort_values('score', ascending=False)

import os
os.makedirs("/content/Machine-Learning-Internship/work/outputs", exist_ok=True)
output_path = os.path.join("/content/Machine-Learning-Internship/work/outputs/baseline_action_score.csv")
ranked_queue.to_csv(output_path, index=False)

print(f"Saved {len(ranked_queue)} rows to {output_path}")
ranked_queue.head(20)


Saved 139747 rows to /content/Machine-Learning-Internship/work/outputs/baseline_action_score.csv


,content_hash_id,client_hash_id,feat_impressions,feat_clicks,position_first_half,position_second_half,position_change,feat_days_active,feat_ctr,declining_flag,volume_bucket,position_slip_bucket,high_volume,position_slipped,score,reason_code,action
29551,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,83772.0,0.0,8.607910,15.567007,6.959097,15.0,0.000000,True,very_high,slipped,1,1,83772.0,traffic_position_decline,review
66466,content_82e35c4845e6c391,client_20259bd6705d81d4,70169.0,29.0,18.269589,26.579565,8.309976,15.0,0.000413,False,very_high,slipped,1,1,70169.0,traffic_position_decline,review
17691,content_65c75874a23fca87,client_23a62021009f63c4,55680.0,15.0,9.013531,17.845249,8.831718,15.0,0.000269,True,very_high,slipped,1,1,55680.0,traffic_position_decline,review
4274,content_62673eea26c31c17,client_65de48885f4ef01b,49386.0,35.0,5.763388,7.850813,2.087425,15.0,0.000709,True,very_high,slipped,1,1,49386.0,traffic_position_decline,review
52638,content_36fc1ee501ec072d,client_62f4a7e64f5e0096,46199.0,11.0,5.241369,7.597220,2.355851,15.0,0.000238,True,very_high,slipped,1,1,46199.0,traffic_position_decline,review
25538,content_afcca85076bb17d3,client_20259bd6705d81d4,40473.0,294.0,2.567940,4.743063,2.175123,15.0,0.007264,False,very_high,slipped,1,1,40473.0,traffic_position_decline,review
66425,content_89c10d52fc81ac39,client_20259bd6705d81d4,35561.0,126.0,21.190152,23.590707,2.400555,15.0,0.003543,False,very_high,slipped,1,1,35561.0,traffic_position_decline,review
50127,content_6f50bf2780b5d040,client_23a62021009f63c4,34873.0,140.0,22.082737,29.669295,7.586558,15.0,0.004015,False,very_high,slipped,1,1,34873.0,traffic_position_decline,review
54780,content_661a7734f691bef5,client_23a62021009f63c4,33724.0,36.0,22.250868,25.424082,3.173214,15.0,0.001067,False,very_high,slipped,1,1,33724.0,traffic_position_decline,review
25310,content_653bbcddf2314227,client_20259bd6705d81d4,32902.0,13.0,10.962283,25.312328,14.350046,15.0,0.000395,True,very_high,slipped,1,1,32902.0,traffic_position_decline,review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**1.** `content_9c057b66c30a3abb` **action:** `review`, **reason:** `traffic_position_decline`, **83772** impressions and decline in position from **8.60** to **15.56**, **Confidence: High** `declining_flag = True` the position slip (6.96) is well past the threshold, and the page has full data coverage (15/15 active days). **What would make it wrong:** My rule only checks first-half impressions and position-slip. It doesn't verify whether second-half impressions also stayed high. If this page's impressions actually grew despite the position drop, flagging it as declining would be wrong.

**2.** `content_82e35c4845e6c391` **action:** `review`, **reason:** `traffic_position_decline`, **70169** impressions and a decline in position from **18.26** to **26.57**, **Confidence: Low** `declining_flag = False` the rule ranked it highly but my own label disagrees, **What would make it wrong:** the `declining_flag = False`, meaning that the rule ranked it for review even though it is not declining, meaning that the underlying traffic may not have dropped meaningfully.

**3.** `content_65c75874a23fca87` **action:** `review`, **reason:** `traffic_position_decline`, **55680** impressions and decline in position from **9.01** to **17.85**, **Confidence: High** `declining_flag = True` and the position slip is well past the threshold, **What would make it wrong:** My rule only checks first-half impressions and position-slip, not whether second-half impressions also stayed high.

**4.** `content_62673eea26c31c17` **action:** `review`, **reason:** `traffic_position_decline`, **49386** impressions and decline in position from **5.76** to **7.85**, **Confidence: Medium** `declining_flag = True` but the position slip barely crosses the threshold, **What would make it wrong:** The slip barely crossing the threshold, combined with the rule not using second-half impressions as an input, means this could be a page with a seasonal dip that trends back up in the next 15 days.

**5.** `content_36fc1ee501ec072d` **action:** `review`, **reason:** `traffic_position_decline`, **46199** impressions and decline in position from **5.24** to **7.60**, **Confidence: Medium** `declining_flag = True` but the position slip barely crosses the threshold, **What would make it wrong:** Same reasoning as above — a barely-crossing slip, with no check on second-half impressions, could reflect a temporary dip rather than a sustained decline.

**6.** `content_afcca85076bb17d3` **action:** `review`, **reason:** `traffic_position_decline`, **40473** impressions and decline in position from **2.57** to **4.74**, **Confidence: Low** `declining_flag = False` plus the position slip barely crosses the threshold, **What would make it wrong:** My rule disagrees with the label I created, which could mean the page is more stable than declining, this could be a minor shift that trends upward in the next 15 days.

**7.** `content_89c10d52fc81ac39` **action:** `review`, **reason:** `traffic_position_decline`, **35561** impressions and decline in position from **21.19** to **23.59**, **Confidence: Low** `declining_flag = False` plus the position slip barely crosses the threshold, **What would make it wrong:** My rule disagrees with the label I created, which could mean the page is more stable than declining, a minor shift that may trend upward in the next 15 days.

**8.** `content_6f50bf2780b5d040` **action:** `review`, **reason:** `traffic_position_decline`, **34873** impressions and decline in position from **22.08** to **29.67**, **Confidence: Low** `declining_flag = False` the rule ranked it highly but my own label disagrees, **What would make it wrong:** My rule only checks first-half impressions and position-slip. If this page's impressions actually grew despite the position drop, flagging it as declining would be wrong.

**9.** `content_661a7734f691bef5` **action:** `review`, **reason:** `traffic_position_decline`, **33724** impressions and decline in position from **22.25** to **25.42**, **Confidence: Low** `declining_flag = False` plus the position slip only barely crosses the threshold, **What would make it wrong:** My rule disagrees with the label I created, which could mean the page is more stable than declining — a minor shift that may trend upward in the next 15 days.

**10.** `content_653bbcddf2314227` **action:** `review`, **reason:** `traffic_position_decline`, **32902** impressions and decline in position from **10.96** to **25.31**, **Confidence: High** `declining_flag = True` plus a huge position slip, **What would make it wrong:** My rule only checks first-half impressions and position-slip. If this page's impressions actually grew despite the position drop, flagging it as declining would be wrong.

**11.** `content_3c20261c0a7c01be` **action:** `review`, **reason:** `traffic_position_decline`, **31994** impressions and decline in position from **5.03** to **7.38**, **Confidence: Low** `declining_flag = False` plus the position slip only barely crosses the threshold, **What would make it wrong:** My rule disagrees with the label I created, which could mean the page is more stable than declining, a minor shift that may trend upward in the next 15 days.

**12.** `content_73aa61dcedebbf30` **action:** `review`, **reason:** `traffic_position_decline`, **31949** impressions and decline in position from **42.40** to **48.80**, **Confidence: Low** `declining_flag = False` the rule ranked it highly but my own label disagrees, **What would make it wrong:** My rule only checks first-half impressions and position-slip. If this page's impressions actually grew despite the position drop, flagging it as declining would be wrong.

**13.** `content_c943bda7971174cd` **action:** `review`, **reason:** `traffic_position_decline`, **31918** impressions and decline in position from **41.55** to **45.32**, **Confidence: Medium** `declining_flag = True` and the starting position was already weak (41.55) with a non-marginal further drop, **What would make it wrong:** My rule doesn't check the change in impressions, it only uses first-half impressions and position-slip. This could be a seasonal dip that trends back upward in the next 15 days.

**14.** `content_f8c0566c8f017176` **action:** `review`, **reason:** `traffic_position_decline`, **31841** impressions and decline in position from **31.33** to **33.58**, **Confidence: Medium** `declining_flag = True` but the position slip barely crosses the threshold, **What would make it wrong:** This could also be a seasonal dip rather than a sustained decline, and may trend upward again in the next 15 days.

**15.** `content_59c5cc86fe3744bf` **action:** `review`, **reason:** `traffic_position_decline`, **31458** impressions and decline in position from **34.55** to **37.09**, **Confidence: Low** `declining_flag = False`, my rule disagrees with the label I designed, **What would make it wrong:** the `declining_flag = False` means the rule ranked it for review even though it is not declining, the underlying traffic may not have dropped meaningfully.

**16.** `content_fa3da71588177d93` **action:** `review`, **reason:** `traffic_position_decline`, **27605** impressions and decline in position from **27.33** to **29.96**, **Confidence: Medium** `declining_flag = True` but the position slip barely crosses the threshold, **What would make it wrong:** The slip barely crossing the threshold, combined with the rule not using second-half impressions as an input, means this could be a seasonal dip that trends back up in the next 15 days.

**17.** `content_a3a1317f7c2bc3dd` **action:** `review`, **reason:** `traffic_position_decline`, **27487** impressions and decline in position from **29.85** to **33.42**, **Confidence: Low** `declining_flag = False`, **What would make it wrong:** My rule disagrees with the label I created, which could mean the page is more stable than declining, rather than genuinely losing visibility.

**18.** `content_8d75e0387a0b4c23` **action:** `review`, **reason:** `traffic_position_decline`, **27385** impressions and decline in position from **22.95** to **26.47**, **Confidence: High** `declining_flag = True` plus a meaningful position slip, **What would make it wrong:** My rule only checks first-half impressions and position-slip. If this page's impressions actually grew despite the position drop, flagging it as declining would be wrong.

**19.** `content_9c78d37a0dc8dc50` **action:** `review`, **reason:** `traffic_position_decline`, **27378** impressions and decline in position from **35.35** to **39.40**, **Confidence: High** `declining_flag = True` plus a meaningful position slip, **What would make it wrong:** My rule only checks first-half impressions and position-slip. If this page's impressions actually grew despite the position drop, flagging it as declining would be wrong.

**20.** `content_67b87ba1ac3d0798` **action:** `review`, **reason:** `traffic_position_decline`, **27139** impressions and decline in position from **8.01** to **12.78**, **Confidence: Low** `declining_flag = False`, **What would make it wrong:** My rule disagrees with the label I created, which could mean the page is more stable than declining rather than genuinely losing visibility.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks (rule and label disagree):**
Roughly half of my top 20 (10 out of 20 rows) show `declining_flag = False` despite scoring highly on my rule, meaning the rule flagged them as high-priority review candidates, but my own label says they did not actually meet the -20% impression-drop criteria. Examples:
**2.** `content_82e35c4845e6c391`, **8.** `content_6f50bf2780b5d040`, **12.** `content_73aa61dcedebbf30`.

This is the most important weakness in the current rule: because it only checks first-half impressions and position-slip, it has no way to confirm whether second-half impressions actually dropped too. A page's position can slip while its impressions stay flat or even grow (e.g. more total searches for that topic, offsetting a worse average rank). My rule cannot tell these cases apart from genuine decline.

**Low-margin correct picks (rule and label agree, but weakly):**
**4.** `content_62673eea26c31c17` and **14.** `content_f8c0566c8f017176` show `declining_flag = True`, so the label agrees with the rule but the position slip in both cases barely crosses my >2 threshold. These are technically correct picks, but on a fragile margin: a slightly different threshold choice could have excluded them, and it's plausible they reflect noise rather than a
clear, sustained decline.

**Leakage Check:**
My rule uses only `feat_impressions`, `position_first_half`, and `position_second_half`, all
computed from raw daily GSC data with the `gsc_data_available` filter applied, none of these are FlyRank product flags (like `health_score` or `priority_score`), and none of them are the label itself or derived from it. `declining_flag` and `trend_pct`-equivalent logic were never used as inputs to the score. They were used only afterward, to check agreement with the rule's
picks (as shown in the weak-picks analysis above). Both halves of the month were used exactly as intended: first-half data to build the rule's inputs, second-half data only to build the label being compared against, no second-half information leaked into the rule's own score.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.